# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
# Load the raw data (latin-1 encoding avoids a UnicodeDecodeError on accented characters)
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)

print(df.shape)
df.info()

(88889, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# Normalize the Make column first (casing, punctuation, legal suffixes, spelling variants)
# so that "CESSNA", "Cessna" and "Cessna Aircraft Co" become a single make.
df['Make'] = (df['Make'].str.strip().str.title()
              .str.replace(r'[.,]', '', regex=True)
              .str.replace(r'\b(Inc|Corp|Corporation|Co|Company|Aircraft|Acft|Industrie|Industries|Ind|Mfg|Llc|Gmbh|Aviation)\b',
                           '', regex=True)
              .str.replace(r'\s+', ' ', regex=True).str.strip()
              .replace({'Dehavilland': 'De Havilland'}))

# Aircraft.Category is missing for ~64% of rows (mostly older records where it was not recorded).
print(df['Aircraft.Category'].value_counts(dropna=False).head(10))

# Imputation: a missing category is inferred from the Make.
# If more than half of a make's labeled records are 'Airplane', its unlabeled records are kept as airplanes
# (Cessna, Piper, ...), while helicopter makes (Bell, Robinson, ...) are excluded.
labeled = df.dropna(subset=['Aircraft.Category'])
share_airplane = (labeled['Aircraft.Category'] == 'Airplane').groupby(labeled['Make']).mean()
airplane_makes = share_airplane[share_airplane > 0.5].index

is_airplane = (df['Aircraft.Category'] == 'Airplane') | \
              (df['Aircraft.Category'].isna() & df['Make'].isin(airplane_makes))
df = df[is_airplane]
print('Airplanes (labeled or inferred):', df.shape)

# The client is only interested in professional builds
df = df[df['Amateur.Built'] == 'No']
print('Professional builds:', df.shape)

# Max lifetime of 40 years: keep events from 1983 onwards
df['Event.Date'] = pd.to_datetime(df['Event.Date'])
df = df[df['Event.Date'] >= '1983-01-01']
print('From 1983 onwards:', df.shape)

Aircraft.Category
NaN                  56602
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
Name: count, dtype: int64
Airplanes (labeled or inferred): (74002, 31)
Professional builds: (69459, 31)
From 1983 onwards: (66458, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']
print(df[injury_cols].isna().sum())

# Assumption: a blank injury count means 0 people in that category.
df[injury_cols] = df[injury_cols].fillna(0)

# Estimated number of people on board = sum of the four injury categories
df['Total.Passengers'] = df[injury_cols].sum(axis=1)

# Records with no person recorded cannot yield a rate: drop them
print('Records with 0 passengers recorded:', (df['Total.Passengers'] == 0).sum())
df = df[df['Total.Passengers'] > 0]

# Key metric: fraction of people on board who were seriously or fatally injured (between 0 and 1)
df['Fatal.Serious.Frac'] = (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers']
df['Fatal.Serious.Frac'].describe()

Total.Fatal.Injuries      8562
Total.Serious.Injuries    9489
Total.Minor.Injuries      9012
Total.Uninjured           4047
dtype: int64
Records with 0 passengers recorded: 1126


count    65332.000000
mean         0.269676
std          0.428318
min          0.000000
25%          0.000000
50%          0.000000
75%          0.666667
max          1.000000
Name: Fatal.Serious.Frac, dtype: float64

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
print(df['Aircraft.damage'].value_counts(dropna=False))

# 'Unknown' is a placeholder for a missing value
df['Aircraft.damage'] = df['Aircraft.damage'].replace('Unknown', np.nan)

# Without damage information the record cannot be used for the destruction metric: drop it
df = df.dropna(subset=['Aircraft.damage'])

# Derived column: 1 if the aircraft was destroyed, 0 otherwise.
# Its mean over a group is the destruction rate of that group.
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)
df['Destroyed'].value_counts()

Aircraft.damage
Substantial    47925
Destroyed      13300
NaN             2021
Minor           2012
Unknown           74
Name: count, dtype: int64


Destroyed
0    49937
1    13300
Name: count, dtype: int64

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
# Cleaning tasks identified for Make (executed in the filtering step above, before inferring categories):
# 1. Inconsistent casing ('CESSNA' vs 'Cessna') -> .str.title()
# 2. Punctuation and legal suffixes ('Air Tractor Inc', 'Cirrus Design Corp') -> regex removal
# 3. Spelling variants ('Dehavilland' vs 'De Havilland') -> explicit replacement
# 4. Leading/trailing/multiple spaces -> .str.strip() and whitespace regex
print('Number of distinct makes:', df['Make'].nunique())

# Keep makes with a reasonable number of records (>= 50) so that rates are statistically robust
make_counts = df['Make'].value_counts()
df = df[df['Make'].isin(make_counts[make_counts >= 50].index)]
print('Makes kept:', df['Make'].nunique(), '| rows:', df.shape[0])
make_counts.head(20)

Number of distinct makes: 903
Makes kept: 64 | rows: 59827


Make
Cessna               25434
Piper                13980
Beech                 5009
Boeing                1302
Mooney                1293
Grumman               1056
Bellanca               974
Air Tractor            890
Aeronca                605
Maule                  570
Champion               501
De Havilland           460
Stinson                421
Aero Commander         405
Mcdonnell Douglas      401
Luscombe               386
Taylorcraft            375
North American         366
Rockwell               334
Cirrus Design          309
Name: count, dtype: int64

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
df = df.dropna(subset=['Model'])
df['Model'] = df['Model'].str.strip().str.upper()

# Model labels are not unique to a make (several makes can use the same code)
models_per_label = df.groupby('Model')['Make'].nunique()
print('Model labels shared by several makes:', (models_per_label > 1).sum())

# Derived unique identifier for a plane type: Make + Model
df['Make.Model'] = df['Make'] + ' ' + df['Model']
df['Make.Model'].value_counts().head(10)

Model labels shared by several makes: 279


Make.Model
Cessna 152         2211
Cessna 172         1614
Cessna 172N        1080
Piper PA-28-140     857
Cessna 172M         755
Cessna 150          742
Cessna 172P         661
Cessna 182          608
Cessna 180          593
Cessna 150M         550
Name: count, dtype: int64

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
for col in ['Engine.Type', 'Weather.Condition', 'Number.of.Engines',
            'Purpose.of.flight', 'Broad.phase.of.flight']:
    print(df[col].value_counts(dropna=False).head(12), '\n')

# Weather.Condition: 'UNK' / 'Unk' are placeholders for missing values; harmonize casing
df['Weather.Condition'] = df['Weather.Condition'].str.upper().replace('UNK', np.nan)

# Engine.Type: 'Unknown', 'UNK', 'NONE' and 'LR' are placeholders or meaningless codes
df['Engine.Type'] = df['Engine.Type'].replace({'Unknown': np.nan, 'UNK': np.nan, 'NONE': np.nan, 'LR': np.nan})

# Number.of.Engines: 0 engines is not plausible for a powered airplane -> placeholder for missing
df.loc[df['Number.of.Engines'] == 0, 'Number.of.Engines'] = np.nan

# Purpose.of.flight and Broad.phase.of.flight: 'Unknown' is a placeholder for missing
df['Purpose.of.flight'] = df['Purpose.of.flight'].replace('Unknown', np.nan)
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].replace('Unknown', np.nan)

Engine.Type
Reciprocating      51861
NaN                 3037
Turbo Prop          2604
Turbo Fan           1327
Unknown              502
Turbo Jet            377
Turbo Shaft          104
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64 

Weather.Condition
VMC    52519
IMC     4791
NaN     1810
UNK      554
Unk      140
Name: count, dtype: int64 

Number.of.Engines
1.0    48719
2.0     8068
NaN     2349
4.0      283
3.0      278
0.0      117
Name: count, dtype: int64 

Purpose.of.flight
Personal               35051
Instructional           8076
Unknown                 3905
Aerial Application      3311
Business                2951
NaN                     2791
Positioning             1013
Ferry                    553
Other Work Use           526
Aerial Observation       386
Executive/corporate      345
Public Aircraft          339
Name: count, dtype: int64 

Broad.phase.of.flight
NaN            16137
Landing        11995
Takeoff         8941
Cruise          7144


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
non_null = df.notna().sum()
print(non_null.sort_values())

# Keep columns with more than 20,000 non-null values
cols_to_drop = non_null[non_null <= 20000].index
print('Dropped columns:', list(cols_to_drop))
df = df.drop(columns=cols_to_drop)
df.shape

Schedule                   7245
Air.carrier               10037
FAR.Description           17433
Aircraft.Category         17558
Longitude                 22202
Latitude                  22208
Airport.Code              36522
Airport.Name              38215
Broad.phase.of.flight     43312
Publication.Date          49449
Purpose.of.flight         53118
Engine.Type               56274
Report.Status             56699
Weather.Condition         57310
Number.of.Engines         57348
Registration.Number       59157
Country                   59722
Location                  59786
Total.Uninjured           59814
Total.Minor.Injuries      59814
Total.Serious.Injuries    59814
Total.Passengers          59814
Fatal.Serious.Frac        59814
Event.Id                  59814
Destroyed                 59814
Amateur.Built             59814
Model                     59814
Make                      59814
Aircraft.damage           59814
Injury.Severity           59814
Event.Date                59814
Accident

(59814, 31)

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
df.to_csv('data/aviation_clean.csv', index=False)
print('Saved:', df.shape)

Saved: (59814, 31)
